In [ ]:
# 1. Montar Google Drive (seguro e idempotente)
from google.colab import drive
import os, subprocess, warnings
warnings.filterwarnings("ignore")

MNT = "/content/drive"
if not os.path.ismount(MNT):
    if os.path.exists(MNT) and os.listdir(MNT):
        try:
            subprocess.run(["fusermount", "-u", MNT], check=False)
        except Exception:
            pass
        !rm -rf /content/drive/*
    drive.mount(MNT, force_remount=True)

ROOT = "MyDrive" if os.path.exists("/content/drive/MyDrive") else "MeuDrive"
BASE_PATH = f"/content/drive/{ROOT}/TCC_USP"
PROC_PATH = os.path.join(BASE_PATH, "data_processed")

print("PROC_PATH:", PROC_PATH)

PROC_PATH: /content/drive/MyDrive/TCC_USP/data_processed


In [ ]:
# 2. Carregar dados processados
import pandas as pd, numpy as np

ibov_file = os.path.join(PROC_PATH, "ibovespa_clean.csv")
news_file = os.path.join(PROC_PATH, "noticias_real_clean.csv")

df_ibov = pd.read_csv(ibov_file)
df_news = pd.read_csv(news_file)

# Normalizar datas
if "date" in df_ibov.columns: df_ibov = df_ibov.rename(columns={"date":"data"})
if "date" in df_news.columns: df_news = df_news.rename(columns={"date":"data"})

df_ibov["data"] = pd.to_datetime(df_ibov["data"])
df_news["data"] = pd.to_datetime(df_news["data"])

# Garantir coluna clean_text
if "clean_text" not in df_news.columns:
    if "texto" in df_news.columns:
        df_news["clean_text"] = df_news["texto"].fillna("")
    else:
        df_news["clean_text"] = ""

print("IBOV:", df_ibov.shape, "| NEWS:", df_news.shape)
print("Datas IBOV:", df_ibov["data"].min(), "->", df_ibov["data"].max())
print("Datas NEWS:", df_news["data"].min(), "->", df_news["data"].max())

IBOV: (20, 2) | NEWS: (81, 6)
Datas IBOV: 2024-01-02 00:00:00 -> 2024-01-29 00:00:00
Datas NEWS: 2025-09-27 00:00:00 -> 2025-09-27 00:00:00


In [ ]:
# 3. Agregar textos por dia e criar alvo do IBOV
# Agrega todas as notícias do mesmo dia
df_text = df_news.groupby("data", as_index=False)["clean_text"].apply(
    lambda s: " ".join([str(x) for x in s if str(x).strip()])
)

df_ibov = df_ibov.sort_values("data").reset_index(drop=True)
df_ibov["ret1"] = df_ibov["close"].pct_change().shift(-1)
df_ibov["y"] = (df_ibov["ret1"] > 0).astype(int)
df_target = df_ibov.dropna(subset=["ret1"]).copy()

# Merge
df = pd.merge(df_target[["data","close","y"]], df_text, on="data", how="inner").sort_values("data")

# ⚠️ Fallback dummy se não houver dados
if df.shape[0] == 0:
    print("⚠️ Nenhuma interseção entre IBOV e notícias. Criando dataset dummy para teste.")
    df = pd.DataFrame({
        "data": pd.date_range("2024-01-02", periods=8, freq="D"),
        "close": [130000,130200,130100,130300,130400,130500,130600,130700],
        "y": [0,1,0,1,1,0,1,0],
        "clean_text": [
            "mercado alta otimismo",
            "queda dolar pressao",
            "petrobras avanco dividendos",
            "ibovespa estabilidade noticias",
            "crescimento economia brasil",
            "inflacao preocupa investidores",
            "mercado futuro reage positivo",
            "politica monetaria em foco"
        ]
    })

print("Após merge/dummy:", df.shape)
display(df.head())

⚠️ Nenhuma interseção entre IBOV e notícias. Criando dataset dummy para teste.
Após merge/dummy: (8, 4)


,data,close,y,clean_text
0,2024-01-02,130000,0,mercado alta otimismo
1,2024-01-03,130200,1,queda dolar pressao
2,2024-01-04,130100,0,petrobras avanco dividendos
3,2024-01-05,130300,1,ibovespa estabilidade noticias
4,2024-01-06,130400,1,crescimento economia brasil


In [ ]:
# 4. TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

docs = df["clean_text"].fillna("").replace("", "vazio").tolist()

vectorizer = TfidfVectorizer(
    min_df=1,
    max_df=1.0,
    ngram_range=(1,2),
    strip_accents="unicode",
    lowercase=True,
    max_features=500
)

X = vectorizer.fit_transform(docs)
y = df["y"].astype(int).reset_index(drop=True)

print("Matriz TF-IDF:", X.shape, "| y:", y.shape)

Matriz TF-IDF: (8, 43) | y: (8,)


In [ ]:
# 5. Avaliação adaptativa
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

def evaluate_timeaware(X, y, models):
    n = X.shape[0]
    results = {}

    if n < 2:
        print(f"⚠️ Dataset com {n} amostra(s): impossível treinar.")
        return results

    # Ajuste automático de splits
    if n >= 30:
        n_splits = 5
    elif n >= 10:
        n_splits = 3
    elif n >= 5:
        n_splits = 2
    else:
        n_splits = 0

    if n_splits >= 2:
        tscv = TimeSeriesSplit(n_splits=n_splits)
        for name, model in models.items():
            aucs, mdas = [], []
            for tr, te in tscv.split(np.arange(n)):
                model.fit(X[tr], y.iloc[tr])
                proba = model.predict_proba(X[te])[:,1]
                pred  = (proba > 0.5).astype(int)
                aucs.append(roc_auc_score(y.iloc[te], proba))
                mdas.append(accuracy_score(y.iloc[te], pred))
            results[name] = {"AUC": float(np.mean(aucs)), "MDA": float(np.mean(mdas))}
        print(f"✅ Avaliação com TimeSeriesSplit (n_splits={n_splits}, n={n})")
    else:
        print(f"⚠️ Poucas amostras (n={n}). Usando holdout 50/50.")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.5, shuffle=False
        )
        for name, model in models.items():
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_test)[:,1]
            pred  = (proba > 0.5).astype(int)
            results[name] = {
                "AUC": float(roc_auc_score(y_test, proba)) if len(np.unique(y_test)) > 1 else float("nan"),
                "MDA": float(accuracy_score(y_test, pred))
            }
    return results

In [ ]:
# 6. Modelos e execução
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42)
}

try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42
    )
except:
    print("ℹ️ XGBoost não disponível, seguindo sem ele.")

results = evaluate_timeaware(X, y, models)
print("\nResultados médios (TF-IDF real/dummy):")
print(results)

✅ Avaliação com TimeSeriesSplit (n_splits=2, n=8)

Resultados médios (TF-IDF real/dummy):
{'Logistic': {'AUC': 0.25, 'MDA': 0.5}, 'RandomForest': {'AUC': 0.25, 'MDA': 0.5}, 'XGBoost': {'AUC': 0.5, 'MDA': 0.5}}
